In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import datetime as dt

from src.data_preparation.input_preparation import prepare_input

In [35]:
df = pd.read_csv("ETT-small/ETTh1.csv")
df_prepped = prepare_input(df)

# Only take columns with high correlation with the target variable (OT)
feature_columns = ['MUFL', 'MULL', 'LUFL']
df_model_features = df_prepped[feature_columns + ['date', 'OT']]

df = df_model_features

In [36]:
test_period_cutoff = dt.datetime(2017, 9, 1, 0, 0, 0)

df_train = df[df['date'] < test_period_cutoff]
df_test = df[df['date'] >= test_period_cutoff]

X_train = df_train.drop(columns=['date', 'OT'])
y_train = df_train['OT']

X_test = df_test.drop(columns=['date', 'OT'])
y_test = df_test['OT']

In [38]:
from src.model.arima import fit_arima
order = (1, 0, 1)
seasonal_order = (1, 1, 0, 24)
arima_model = fit_arima(y_train, X_train, order, seasonal_order)
# arima_model.summary()

In [52]:
y_pred = arima_model.get_forecast(steps = len(y_test), exog=X_test).predicted_mean
SSE = ((y_pred - y_test) ** 2).sum()
WMAPE = (abs(y_pred - y_test).sum() / y_test.sum()) * 100
print(f"Arima sum of Squared Errors (SSE): {SSE}")
print(f"Arima Weighted Mean Absolute Percentage Error (WMAPE): {WMAPE}%")

Arima sum of Squared Errors (SSE): 179934.49940337788
Arima Weighted Mean Absolute Percentage Error (WMAPE): 54.042852187077%
